## 6.2. Image Convolution

### 6.2.1 Cross-Correlation

strictly speacking, the convolution layer in ComputerScience is actually 'cross-correlation';

output size given input size $n_h × n_w$ and kernel size $k_h × k_w$:
$$
(n_h - k_h + 1) × (n_w - k_w + 1)
$$

In [2]:
import torch
from torch import nn
from d2l import torch as d2l

def corr2d(X, K): #@save
  """compute 2d correlation"""
  h, w = K.shape
  Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
  for i in range(Y.shape[0]):
    for j in range(Y.shape[1]):
      Y[i, j] = (X[i:i + h, j:j + w] * K).sum()
  return Y

In [3]:
X = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
K = torch.tensor([[0.0, 1.0], [2.0, 3.0]])
corr2d(X, K)

tensor([[19., 25.],
        [37., 43.]])

### 6.2.2 Convolutional Layer

In [4]:
class Conv2D(nn.Module):
  def __init__(self, kernel_size):
    super().__init__()
    self.weight = nn.Parameter(torch.rand(kernel_size))
    self.bias = nn.Parameter(torch.zeros(1))
    
  def forward(self, x):
    return corr2d(x, self.weight) + self.bias

### 6.2.3 Image Edge Dection

In [5]:
X = torch.ones((6, 8))
X[:, 2:6] = 0
X

tensor([[1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.]])

In [7]:
K = torch.tensor([[1.0, -1.0]])

In [8]:
Y = corr2d(X, K)
Y

tensor([[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.]])

In [9]:
corr2d(X.t(), K)

tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])

### 6.2.4 Learning Convolution Kernel

In [10]:
# construct an 2d convolutional layer, 1 output channel and (1,2)-sized conv kernel
conv2d = nn.Conv2d(1,1,kernel_size=(1,2), bias=False)

# 4d Input and Output(batch size, channel, height, weight)
# batch_size = channel = 1
X = X.reshape((1, 1, 6, 8))
Y = Y.reshape((1, 1, 6, 7))
lr = 3e-2 # learning rate

for i in range(10):
  Y_hat = conv2d(X)
  l = (Y_hat - Y) ** 2
  conv2d.zero_grad()
  l.sum().backward()
  # update conv kernel
  conv2d.weight.data[:] -= lr * conv2d.weight.grad
  if (i + 1) % 2 == 0:
    print(f'epoch {i + 1}, loss {l.sum():.3f}')

epoch 2, loss 0.713
epoch 4, loss 0.147
epoch 6, loss 0.036
epoch 8, loss 0.011
epoch 10, loss 0.004


In [11]:
conv2d.weight.data.reshape((1, 2))

tensor([[ 0.9901, -1.0018]])

### 6.2.5 Cross-Correlation and Convolution

### 6.2.6 Feature Map and Receptive Field